# Modelo BYM2 por comuna (revisión del 1-oct-2026)

Responde los comentarios 10, 11 y 39: lleva la pregunta espacial al nivel comunal y estima la fracción de la varianza comunal con estructura espacial (parámetro ρ; Riebler et al., 2016).

**Requiere** el notebook 06 ejecutado (`bayes_escuelas_mate.csv`).

**Lectura de ρ.** Cercano a 1: las comunas vecinas se parecen (estructura espacial). Cercano a 0: la variación comunal es independiente del territorio. Un intervalo que cubre casi todo [0, 1] indica que los datos no permiten distinguirlo.

> **Revisión del 24-oct-2026 (auditoría de cuadernos y código).** `bym2.py` tenía mal el **factor de escala** del efecto espacial: lo calculaba con la inversa de Q más una constante en la diagonal, y esa constante convertía la dirección sin información del ICAR en una varianza enorme. El registro lo mostraba: "factor de escala 8962,324", cuando el valor correcto ronda 0,78. Con ese factor la parte espacial quedaba anulada y ρ no podía estimarse: por eso su intervalo cubría [0,01; 0,99].

La versión 1.1.0 sigue a Riebler et al. (2016) y Freni-Sterrantino et al. (2018): el grafo de 5 vecinos tiene **3 componentes conexas** (10, 61 y 259 comunas), cada una recibe su propio factor de escala (0,15; 0,67; 0,86) y su propia restricción de suma cero. Además usa cuatro cadenas de 1.500 + 1.500 iteraciones (unos 5 minutos por variable), porque con dos cadenas el R-hat de ρ quedaba en 1,01–1,02.

**Salida esperada:**

| | Fracción comunal de la varianza | ρ | R-hat de ρ |
|---|---|---|---|
| Caída | 0,045 [0,022; 0,085] | 0,86 [0,27; 1,00] | 1,009 |
| Recuperación | 0,024 [0,008; 0,054] | 0,68 [0,07; 1,00] | 1,010 |

Antes: fracción 0,14 [0,03; 0,84] y 0,09 [0,02; 0,73]; ρ 0,78 [0,01; 0,99] en ambas. La lectura cambia: la comuna explica poco, pero en la caída esa pequeña parte sí tiene estructura espacial.

In [1]:
import sys, warnings
from pathlib import Path
for m in list(sys.modules):
    if m == "simce_panel" or m.startswith("simce_panel."):
        del sys.modules[m]
candidatos = [Path.cwd() / "simce_panel" / "src", Path.cwd().parent / "src", Path.cwd() / "src"]
sys.path.insert(0, str(next(p for p in candidatos if (p / "simce_panel" / "__init__.py").exists())))
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import simce_panel as sp
sp.configurar_logging()
cfg = sp.Config.desde_base(".")

In [2]:
esc = sp.io_datos.leer_csv(cfg.ruta_salida("bayes_escuelas_mate.csv"))
ancho = sp.io_datos.leer_csv(cfg.ruta_salida("simce_rbd_ancho_reducido_2015_2025.csv"))
res = sp.io_datos.leer_csv(cfg.ruta_salida("resultados_mate_2m.csv"))
d = (esc.merge(ancho[["rbd", "comuna_dir", "latitud", "longitud", "cod_grupo_cat", "cod_depe1_cat"]],
               on="rbd", how="left")
        .merge(res[["rbd", "puntaje_pre"]], on="rbd", how="left"))
grafo = sp.bym2.grafo_comunas(d, k=5)
print(grafo["n_comunas"], "comunas;", len(grafo["nodo1"]), "aristas")

17:04:42 | INFO    | simce_panel.bym2 | Grafo: 330 comunas, 1025 aristas, 3 componentes conexos (0 aisladas); factor de escala medio 0.778.


330 comunas; 1025 aristas


## 1. Ajuste

Covariables: grupo socioeconómico, dependencia y puntaje previo. Se usan cuatro cadenas para que el R-hat de ρ sea informativo; la tabla trae `r_hat_rho` y `ess_rho`, porque ρ es el parámetro que mezcla peor (su posterior se acerca al borde superior).

In [3]:
resultados = [sp.bym2.ajustar_bym2(d, col, ["cod_grupo_cat", "cod_depe1_cat", "puntaje_pre"], grafo)
              for col in ("caida_bayes", "recuperacion_bayes")]
pd.DataFrame(resultados)

17:04:43 | INFO    | jax._src.xla_bridge | Unable to initialize backend 'tpu': UNIMPLEMENTED: LoadPjrtPlugin is not implemented on windows yet.


,variable,n,comunas,"rho_[3,50,97]","fraccion_varianza_comunal_[3,50,97]",divergencias,r_hat_max,r_hat_rho,ess_rho,segundos
0,caida_bayes,2861,330,"[0.249, 0.844, 0.999]","[0.022, 0.045, 0.086]",0,1.005,1.004,575,351.5
1,recuperacion_bayes,2861,330,"[0.087, 0.682, 0.998]","[0.009, 0.025, 0.055]",0,1.002,1.002,672,327.2


## 2. Interpretación (corregida el 24-oct-2026)

**Cuánto pesa la comuna.** Poco: cerca del 5 % de la varianza de la caída y del 2 % de la recuperación, una vez descontados GSE, dependencia y puntaje previo.

**Si ese poco tiene geografía.** En la caída, sí: el posterior de ρ se concentra en valores altos y su intervalo excluye los cercanos a cero, de modo que las comunas vecinas se parecen en la parte de la caída que no explica la composición. En la recuperación, ρ sigue siendo indeterminado. Es coherente con el Moran de los residuos del cuaderno 05: una geografía débil en la caída y ninguna clara en la recuperación.

**Cómo citar ρ.** Siempre junto con la fracción comunal: un ρ alto dice qué parte de la variación *comunal* es espacial, no que la geografía pese mucho.

**Limitación y versión definitiva.** Este cuaderno usa como dato las medias a posteriori del modelo de medición, lo que ignora su incertidumbre. La versión definitiva, para Capstone II, integra el efecto BYM2 dentro del modelo de medición del cuaderno 06.